# Stage 3 — Create the 72-hour label

**What:** Turn old request records into a yes/no answer: `1` means the request was not closed within 72 hours; `0` means it closed within 72 hours.

**Why:** The model needs examples with known answers. A 2-hour-old open request has an **unknown** label and cannot be used for training yet.

Examples: closed after 20 hours → `0`; closed after 90 hours → `1`; still open after 5 days → `1`; still open after 2 hours → unknown.

We will read the exact Stage 2 snapshot and record its ID in the output. This is **data lineage**: knowing which raw data produced which training data.

**Time rule:** The API's dates do not include a time zone. We interpret them as New York local time, then convert to UTC before measuring 72 actual hours. Rows with unclear or invalid times are excluded and counted.

In [ ]:
# Cell 2 — Why: find the saved snapshots from Stage 2.
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")
project = Path("/content/drive/MyDrive/nyc311_mlops")
raw_root = project / "raw"
snapshot_ids = sorted(p.name for p in raw_root.iterdir() if p.is_dir())
print("Available snapshot IDs:", snapshot_ids[-5:])

In [ ]:
# Cell 3 — Why: choose one exact snapshot and verify it before use.
import hashlib
import json

snapshot_id = input("Paste one snapshot ID from Cell 2: ").strip()
if snapshot_id not in snapshot_ids:
    raise ValueError("Choose one of the listed snapshot IDs")
snapshot_dir = raw_root / snapshot_id
raw_bytes = (snapshot_dir / "requests.jsonl").read_bytes()
manifest = json.loads((snapshot_dir / "manifest.json").read_text())
if hashlib.sha256(raw_bytes).hexdigest() != manifest["sha256"]:
    raise ValueError("Raw file does not match its manifest")
records = [json.loads(line) for line in raw_bytes.splitlines()]
if len(records) != manifest["row_count"]:
    raise ValueError("Row count does not match the raw manifest")
print("Verified snapshot:", snapshot_id, "with", len(records), "rows")

In [ ]:
# Cell 4 — Why: measure real elapsed hours, including time-zone changes.
import pandas as pd

df = pd.DataFrame(records).reindex(
    columns=["unique_key", "created_date", "closed_date", "complaint_type", "borough"]
)
created_local = pd.to_datetime(df["created_date"], errors="coerce")
closed_local = pd.to_datetime(df["closed_date"], errors="coerce")
created = created_local.dt.tz_localize(
    "America/New_York", ambiguous="NaT", nonexistent="NaT"
).dt.tz_convert("UTC")
closed = closed_local.dt.tz_localize(
    "America/New_York", ambiguous="NaT", nonexistent="NaT"
).dt.tz_convert("UTC")
as_of = pd.Timestamp(manifest["fetched_at_utc"])
print("Source snapshot collected at:", as_of)

In [ ]:
# Cell 5 — Why: label only requests with a known, valid 72-hour outcome.
deadline = created + pd.Timedelta(hours=72)
bad_time = created.isna() | (df["closed_date"].notna() & closed.isna())
bad_time |= closed.notna() & (closed < created)
bad_time |= closed.notna() & (closed > as_of)
closed_early = closed.notna() & (closed <= deadline)
old_enough = deadline <= as_of
eligible = ~bad_time & (closed_early | old_enough)

labels = (~closed_early).astype(int)
training_rows = df.loc[eligible, [
    "unique_key", "created_date", "complaint_type", "borough"
]].copy()
training_rows["label"] = labels.loc[eligible].to_numpy()

print("Bad timestamps excluded:", int(bad_time.sum()))
print("Outcome still unknown:", int((~(closed_early | old_enough) & ~bad_time).sum()))
print("Labeled rows:", len(training_rows))
display(training_rows["label"].value_counts().sort_index())

In [ ]:
# Cell 6 — Why: save training rows with a link to their raw snapshot.
from datetime import datetime, timezone

label_run = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
label_dir = project / "labeled" / label_run
label_dir.mkdir(parents=True, exist_ok=False)
label_bytes = "".join(
    json.dumps(row, sort_keys=True) + "\n"
    for row in training_rows.to_dict(orient="records")
).encode("utf-8")
label_file = label_dir / "training_rows.jsonl"
label_file.write_bytes(label_bytes)

label_manifest = {
    "raw_snapshot_id": snapshot_id,
    "raw_fetched_at_utc": manifest["fetched_at_utc"],
    "label_rule": "1 if not closed within 72 actual hours; else 0",
    "rows": len(training_rows),
    "sha256": hashlib.sha256(label_bytes).hexdigest(),
    "lesson_version": "stage3-v1",
}
(label_dir / "manifest.json").write_text(
    json.dumps(label_manifest, indent=2), encoding="utf-8"
)
if hashlib.sha256(label_file.read_bytes()).hexdigest() != label_manifest["sha256"]:
    raise ValueError("Saved labels failed their checksum check")
print("Saved:", label_file)
print("Raw snapshot used:", snapshot_id)

# Stage 3 takeaway and quiz

**Why this stage mattered:** We turned later outcomes into trustworthy training labels and kept the connection to the raw snapshot. The saved training file contains the inputs and label, but not `closed_date`, so it cannot leak the answer into training.

**Easy quiz:**

1. A request closed after 20 hours. What is its label?
2. A request closed after 90 hours. What is its label?
3. A request is 2 hours old and still open. Can we label it yet?
4. Why is `closed_date` used to make the label but removed from the saved training rows?

Send me the counts from Cell 5, the path from Cell 6, and your answers. We will inspect and validate the training data in Stage 4.